# Evaluate a code-review agent

[Open in Colab](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/use-cases/coding-agent-eval.ipynb)

Will a reviewer find the important defect, and will it leave correct code alone? Run two prompts against the same six labeled snippets. Compare a general completeness score with a finding-specific rubric and a deterministic format check. The snippets are review inputs, not code to execute.

## Install and authenticate

Use a fresh Python 3.12 kernel. OpenAI generates and grades reviews; the kit runs the hosted completeness check with your Future AGI credentials. This notebook uses only synthetic snippets.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                'agent-learning-kit==0.3.7', 'openai==2.54.0'], check=True)

In [ ]:
import os
from getpass import getpass

for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value.strip():
            raise ValueError(f"{name} is required.")
        os.environ[name] = value.strip()
        del value

## Label the cases before generating reviews

Give the scorer a concrete expectation for each finding, including a clean control. The reviewer will see the snippet but not the answer key.

In [ ]:
cases = [
    {"id": "log-privacy", "label": "PRIVACY", "source": "def log_sign_in(logger, password):\n    logger.info(password)",
     "expected": "Logging the password exposes sensitive data. Omit or redact it."},
    {"id": "money", "label": "BUG", "source": 'def discounted(price, pct):\n    return price - price * pct / "100"',
     "expected": "Dividing a number by a string raises TypeError. Use the numeric divisor 100."},
    {"id": "empty-list", "label": "BUG", "source": "def mean_weight(weights):\n    return sum(weights) / len(weights)",
     "expected": "An empty list divides by zero. Define how the function handles empty input."},
    {"id": "repeated-work", "label": "PERFORMANCE", "source": "def has_repeat(tags):\n    return any(tags.count(t) > 1 for t in tags)  # tags: list[str]",
     "expected": "Counting inside the loop can be quadratic. Track seen tags or compare the size of a set."},
    {"id": "naming", "label": "STYLE", "source": "def ShipDays(ZipCode):\n    return 2 if ZipCode.startswith('9') else 5",
     "expected": "The public function and parameter should use snake_case under PEP 8."},
    {"id": "clean", "label": "OK", "source": "def has_stock(quantity: int) -> bool:\n    return quantity > 0",
     "expected": "There is no demonstrated defect; do not invent one."},
]

## Generate reviews

Keep the model and cases fixed. Only the revised prompt states the label and response contract.

In [ ]:
from openai import OpenAI

client = OpenAI()

prompts = {
    "baseline": "Review the Python snippet. State the main issue and suggest a fix, or say it looks good.",
    "revised": ("Review the Python snippet. Use exactly one bracketed label in the whole reply. Put it first: [PRIVACY], "
                "[BUG], [PERFORMANCE], [STYLE] or [OK]. Choose the most important demonstrated issue. "
                "Explain its consequence and recommend a minimal fix in at most two sentences. "
                "For correct code, use [OK] and do not invent defects."),
}


def review(prompt, snippet):
    response = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "system", "content": prompt}, {"role": "user", "content": snippet}],
    )
    text = (response.choices[0].message.content or "").strip()
    if not text:
        raise RuntimeError("The review was empty.")
    return text


reviews = [{**case, "version": version, "review": review(prompt, case["source"])}
           for version, prompt in prompts.items() for case in cases]
print(f"Generated {len(reviews)} reviews.")

## Score what the review actually says

Completeness judges the same review task in both rounds. The custom rubric checks the known finding, not exact wording. The format check treats uppercase bracketed tags as labels. It requires the expected tag at the start and no other label in the reply.

In [ ]:
import json
from fi.evals import evaluate, Evaluator

reader = Evaluator()


def hosted(metric, **inputs):
    result = evaluate(metric, model="turing_small", **inputs)
    if result.status != "completed" or result.score is None:
        raise RuntimeError(f"{metric} returned no score: {result.error or result.reason}")
    eval_id = result.metadata["eval_id"]
    record = reader.get_eval_result(eval_id)["result"]["result"]
    if str(record["eval_id"]) != str(eval_id) or record["name"] != metric or record.get("error_message"):
        raise RuntimeError(f"Could not confirm stored {metric} result.")
    return result


def grade(rubric, **inputs):
    result = evaluate(engine="llm", model="openai/gpt-4o-mini", prompt=rubric, **inputs)
    if result.status != "completed" or result.score is None:
        raise RuntimeError("The rubric judge did not return a score.")
    # The kit serializes a successful custom judgment in reason. A provider or
    # parse failure is not an ordinary low-quality result.
    try:
        detail = json.loads(result.reason)
        if float(detail["score"]) != result.score or not detail["reason"].strip():
            raise ValueError
    except (ValueError, TypeError, KeyError) as exc:
        raise RuntimeError("The rubric judge did not return a usable verdict.") from exc
    return result.score, detail["reason"]

In [ ]:
import re

labels = re.compile(r"\[[A-Z][A-Z0-9_-]*\]")
rubric = ("Compare review with the reference finding and the snippet. Score 1 only if the review "
          "identifies the main issue and an appropriate fix, or correctly leaves the clean control "
          "alone. Ignore cosmetic wording and formatting. Otherwise score 0. Explain your verdict.")
for row in reviews:
    complete = hosted("completeness", input=prompts["baseline"] + "\n" + row["source"], output=row["review"])
    coverage, why = grade(rubric, source=row["source"], expected=row["expected"], review=row["review"])
    expected_label = f"[{row['label']}]"
    label_ok = row["review"].startswith(expected_label) and labels.findall(row["review"]) == [expected_label]
    row.update(completeness=complete.score, eval_id=complete.metadata["eval_id"],
               finding_score=coverage, reason=why, label_ok=label_ok)
    print(f"{row['version']:8} {row['id']:14} completeness={complete.score:.2f} "
          f"finding={coverage:.2f} label_ok={row['label_ok']}")

## Compare prompts and inspect regressions

A correct label does not prove the explanation is right. Review cases where either version missed the finding, especially the clean control.

In [ ]:
import csv
from statistics import mean

for version in prompts:
    group = [r for r in reviews if r["version"] == version]
    print(version, "|", " | ".join(f"{key}: {mean(r[key] for r in group):.2f}"
                                    for key in ("completeness", "finding_score", "label_ok")))
for r in reviews:
    if r["finding_score"] < 1:
        print(f"{r['version']} / {r['id']}: {r['review']}\nJudge: {r['reason']}\n")

with open("code_review_scores.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(reviews[0]))
    writer.writeheader()
    writer.writerows(reviews)

`code_review_scores.csv` keeps the review text, label result, judge reason and hosted evaluation ID together. Six synthetic cases can expose concrete regressions, but do not establish overall reviewer quality. Add held-out cases before adopting either prompt.